# Exit Advisor fine-tuning + evals notebook

This notebook is focused on the **Exit Advisor** only.

Goal:
- build a clean **binary end / not_end** dataset from `sms_conversations.json`
- evaluate the **current prompt-based Exit Advisor** as a baseline
- fine-tune a model for **exit detection**
- test the fine-tuned model on a held-out **20% test split**
- compare accuracy and confusion matrix

Recommended design for the project:
- keep the **Main Agent** as the 3-way router: `continue / schedule / end`
- keep the **Exit Advisor** as a **binary classifier**: `end / not_end`
- keep polite ending wording in code or a small prompt template
- fine-tune only the **detection** part

That keeps the Exit Advisor simple, deterministic, and easy to evaluate.

## Why this notebook structure

Your project instructions explicitly say:
- the system is evaluated on `continue / schedule / end`
- the **Conversation Exit Advisor should be fine-tuned to detect conversation scenarios that are expected to conclude**
- evaluation should use **accuracy** and **confusion matrix** on the labeled dataset.

Also, the uploaded dataset contains labeled recruiter turns, including clear ending cases such as:
- booked interview confirmations
- candidate asks to stop texting
- candidate says they are no longer interested.

So for the Exit Advisor, the cleanest target is:

- `end` → positive class
- `continue` and `schedule` → negative class

This notebook uses **conversation-level splitting** so the same conversation does not leak into both train and test.

In [ ]:
from pathlib import Path
import json
import random
from collections import Counter

import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

DATA_PATH = Path("sms_conversations.json")
assert DATA_PATH.exists(), f"Could not find {DATA_PATH.resolve()}"

with open(DATA_PATH, "r", encoding="utf-8") as f:
    conversations = json.load(f)

len(conversations)

## Build Exit Advisor examples

Important choice:
we create an example only for recruiter turns that:
1. have a label
2. are **not** the first message
3. come **after a candidate message**

Why:
the Exit Advisor should usually be consulted **after the candidate just said something**, not on the recruiter's opening message.

In [ ]:
def build_exit_examples(conversations):
    rows = []

    for conv in conversations:
        turns = conv["turns"]

        for i, turn in enumerate(turns):
            if i == 0:
                continue

            if turn["speaker"] != "recruiter":
                continue

            label = turn.get("label")
            if label not in {"continue", "schedule", "end"}:
                continue

            prev_turn = turns[i - 1]
            if prev_turn["speaker"] != "candidate":
                continue

            history_turns = turns[:i]

            rows.append(
                {
                    "conversation_id": conv["conversation_id"],
                    "turn_id": turn["turn_id"],
                    "next_recruiter_label": label,
                    "binary_label": "end" if label == "end" else "not_end",
                    "history_turns": history_turns,
                    "candidate_last_message": prev_turn["text"],
                    "expected_recruiter_message": turn["text"],
                }
            )

    return pd.DataFrame(rows)

df = build_exit_examples(conversations)
print("examples:", len(df))
print(df["binary_label"].value_counts())
print(df["next_recruiter_label"].value_counts())
df.head(3)

Expected from this dataset:
- 44 Exit Advisor decision points
- 15 `end`
- 29 `not_end`

That is enough for a first supervised fine-tuning pass, though still a small dataset.

In [ ]:
SEED = 42

conversation_ids = sorted(df["conversation_id"].unique().tolist())
rng = random.Random(SEED)
shuffled_ids = conversation_ids[:]
rng.shuffle(shuffled_ids)

test_n = max(1, round(len(shuffled_ids) * 0.20))
test_ids = sorted(shuffled_ids[:test_n])
train_ids = sorted(shuffled_ids[test_n:])

train_df = df[df["conversation_id"].isin(train_ids)].copy().reset_index(drop=True)
test_df = df[df["conversation_id"].isin(test_ids)].copy().reset_index(drop=True)

print("train conversations:", train_ids)
print("test conversations :", test_ids)
print()
print("train size:", len(train_df), Counter(train_df["binary_label"]))
print("test size :", len(test_df), Counter(test_df["binary_label"]))

## Load your current Exit Advisor prompt from `exit_agent.py`

This notebook tries to import your existing instructions from the project code.

If the import fails, paste the **exact current developer instructions** from your `exit_agent.py` into `EXIT_ADVISOR_BASELINE_INSTRUCTIONS`.

That gives you a true baseline starting from your current implementation.

In [ ]:
EXIT_ADVISOR_BASELINE_INSTRUCTIONS = None

possible_imports = [
    ("app.modules.orchestration.exit_agent", "EXIT_ADVISOR_INSTRUCTIONS"),
    ("app.modules.orchestration.exit_agent", "EXIT_AGENT_INSTRUCTIONS"),
    ("app.modules.orchestration.exit_agent", "EXIT_PROMPT"),
    ("app.modules.orchestration.exit_agent", "EXIT_SYSTEM_PROMPT"),
    ("app.modules.orchestration.exit_agent", "EXIT_ADVISOR_PROMPT"),
]

for module_name, attr_name in possible_imports:
    try:
        module = __import__(module_name, fromlist=[attr_name])
        EXIT_ADVISOR_BASELINE_INSTRUCTIONS = getattr(module, attr_name)
        print(f"Loaded prompt from {module_name}.{attr_name}")
        break
    except Exception:
        pass

if EXIT_ADVISOR_BASELINE_INSTRUCTIONS is None:
    EXIT_ADVISOR_BASELINE_INSTRUCTIONS = """
You are the Conversation Exit Advisor for a recruiting chatbot.

Your job is only to decide whether the conversation should conclude now.

Return exactly one label:
- end
- not_end

Choose end only if the candidate clearly wants to stop the conversation,
is no longer interested, asked not to be contacted, or the conversation
has naturally concluded after interview confirmation / mutual closing.

Choose not_end in all other cases, including when the conversation should
continue or move into scheduling.
""".strip()
    print("Could not import exit_agent.py prompt. Using fallback prompt. Paste your current prompt here before running the baseline.")
else:
    print("Prompt preview:")
    print(EXIT_ADVISOR_BASELINE_INSTRUCTIONS[:700])

## Helper functions

We will use the **Responses API** for inference.

The model should output exactly:
- `end`
- `not_end`

This is intentionally minimal. It is easier to evaluate, easier to fine-tune, and easier to plug back into your current agent code.

In [ ]:
from openai import OpenAI

client = OpenAI()

BASELINE_MODEL = "gpt-4.1-mini"

def history_to_text(history_turns):
    lines = []
    for t in history_turns:
        role = t["speaker"].upper()
        lines.append(f"{role}: {t['text']}")
    return "\n".join(lines)

def build_exit_input(history_turns):
    conversation_text = history_to_text(history_turns)
    return (
        "Conversation so far:\n"
        f"{conversation_text}\n\n"
        "Question: Should the recruiter end the conversation now?\n"
        "Respond with exactly one label: end or not_end."
    )

def classify_exit(model_name, instructions, history_turns):
    response = client.responses.create(
        model=model_name,
        input=[
            {"role": "developer", "content": instructions},
            {"role": "user", "content": build_exit_input(history_turns)},
        ],
        temperature=0,
    )
    text = (response.output_text or "").strip().lower()

    if text not in {"end", "not_end"}:
        if "end" in text and "not_end" not in text:
            return "end"
        return "not_end"

    return text

## Baseline eval with the current Exit Advisor prompt

This is the first thing to measure.

OpenAI recommends doing evals first before investing in fine-tuning, and recommends starting with a small but realistic dataset in chat format. Fine-tuning supports the `gpt-4.1` family, expects JSONL in chat-completions format, requires at least 10 examples, and often starts to improve with roughly 50–100 examples, though results vary by task.

In [ ]:
def run_eval(df_eval, model_name, instructions):
    rows = []

    for _, row in df_eval.iterrows():
        pred = classify_exit(
            model_name=model_name,
            instructions=instructions,
            history_turns=row["history_turns"],
        )
        rows.append(
            {
                "conversation_id": row["conversation_id"],
                "turn_id": row["turn_id"],
                "gold": row["binary_label"],
                "pred": pred,
                "next_recruiter_label": row["next_recruiter_label"],
                "candidate_last_message": row["candidate_last_message"],
            }
        )

    pred_df = pd.DataFrame(rows)
    return pred_df

# WARNING:
# This cell makes API calls.
# Uncomment to run.

# baseline_test_results = run_eval(
#     df_eval=test_df,
#     model_name=BASELINE_MODEL,
#     instructions=EXIT_ADVISOR_BASELINE_INSTRUCTIONS,
# )
# baseline_test_results.head()

In [ ]:
def summarize_eval(results_df, title="Eval"):
    y_true = results_df["gold"]
    y_pred = results_df["pred"]

    acc = accuracy_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred, labels=["not_end", "end"])

    print(title)
    print("accuracy:", round(acc, 4))
    print()
    print(classification_report(y_true, y_pred, labels=["not_end", "end"]))

    fig, ax = plt.subplots(figsize=(5, 4))
    ax.imshow(cm)
    ax.set_xticks([0, 1], labels=["not_end", "end"])
    ax.set_yticks([0, 1], labels=["not_end", "end"])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(title)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, cm[i, j], ha="center", va="center")

    plt.tight_layout()
    plt.show()

    return acc, cm

## Inspect baseline mistakes

These are the examples you should look at before fine-tuning.

They tell you what behavior to teach.

In [ ]:
# mistakes = baseline_test_results[baseline_test_results["gold"] != baseline_test_results["pred"]].copy()
# mistakes

## Prepare fine-tuning data

For fine-tuning, we use only the **training conversations**.

Then we split them again into:
- FT train
- FT validation

Format:
each line is one JSON object with `messages`, in chat-completions format.

In [ ]:
ft_train_df, ft_val_df = train_test_split(
    train_df,
    test_size=0.18,
    random_state=SEED,
    stratify=train_df["binary_label"],
)

print("ft_train:", len(ft_train_df), Counter(ft_train_df["binary_label"]))
print("ft_val  :", len(ft_val_df), Counter(ft_val_df["binary_label"]))

In [ ]:
def make_ft_record(row, instructions):
    return {
        "messages": [
            {"role": "developer", "content": instructions},
            {"role": "user", "content": build_exit_input(row["history_turns"])},
            {"role": "assistant", "content": row["binary_label"]},
        ]
    }

train_jsonl_path = Path("exit_advisor_train.jsonl")
val_jsonl_path = Path("exit_advisor_val.jsonl")

with open(train_jsonl_path, "w", encoding="utf-8") as f:
    for _, row in ft_train_df.iterrows():
        f.write(json.dumps(make_ft_record(row, EXIT_ADVISOR_BASELINE_INSTRUCTIONS), ensure_ascii=False) + "\n")

with open(val_jsonl_path, "w", encoding="utf-8") as f:
    for _, row in ft_val_df.iterrows():
        f.write(json.dumps(make_ft_record(row, EXIT_ADVISOR_BASELINE_INSTRUCTIONS), ensure_ascii=False) + "\n")

print(train_jsonl_path.resolve())
print(val_jsonl_path.resolve())

In [ ]:
with open(train_jsonl_path, "r", encoding="utf-8") as f:
    for _ in range(2):
        print(f.readline())

## Upload files and create the fine-tuning job

This section uses the current OpenAI Python SDK pattern:
- upload training / validation files
- create a fine-tuning job
- wait for completion
- use the returned fine-tuned model id

OpenAI's docs also note that:
- supervised fine-tuning is appropriate for classification
- default hyperparameters are a good starting point
- if the model underfits a classification task, increasing epochs slightly can help.

In [ ]:
# WARNING:
# This section makes API calls and may incur cost.

# train_file = client.files.create(
#     file=open(train_jsonl_path, "rb"),
#     purpose="fine-tune",
# )
# val_file = client.files.create(
#     file=open(val_jsonl_path, "rb"),
#     purpose="fine-tune",
# )

# print("train_file:", train_file.id)
# print("val_file  :", val_file.id)

# ft_job = client.fine_tuning.jobs.create(
#     model="gpt-4.1-mini-2025-04-14",
#     training_file=train_file.id,
#     validation_file=val_file.id,
#     suffix="exit-advisor-v1",
# )

# ft_job

In [ ]:
# Poll the fine-tuning job until it finishes.
# Replace FT_JOB_ID with the id returned above.

# FT_JOB_ID = ft_job.id

# import time
# while True:
#     job = client.fine_tuning.jobs.retrieve(FT_JOB_ID)
#     print(job.status)
#     if job.status in {"succeeded", "failed", "cancelled"}:
#         break
#     time.sleep(15)

# job

In [ ]:
# When finished successfully:
# FT_MODEL = job.fine_tuned_model
# print("Fine-tuned model:", FT_MODEL)

## Evaluate the fine-tuned model on the held-out test set

Do **not** evaluate on the training rows.

Only test on the held-out 20% conversation-level split.

In [ ]:
# FT_MODEL = "ft:gpt-4.1-mini-2025-04-14:your-org:exit-advisor-v1:xxxxx"

# ft_test_results = run_eval(
#     df_eval=test_df,
#     model_name=FT_MODEL,
#     instructions=EXIT_ADVISOR_BASELINE_INSTRUCTIONS,
# )

# ft_test_results.head()

In [ ]:
# baseline_acc, baseline_cm = summarize_eval(
#     baseline_test_results,
#     title="Baseline Exit Advisor on Test Set"
# )

# ft_acc, ft_cm = summarize_eval(
#     ft_test_results,
#     title="Fine-Tuned Exit Advisor on Test Set"
# )

# print("baseline accuracy :", baseline_acc)
# print("fine-tuned accuracy:", ft_acc)
# print("delta             :", round(ft_acc - baseline_acc, 4))

## Side-by-side comparison of errors

Look for examples where:
- baseline said `not_end` but FT correctly said `end`
- baseline said `end` too early but FT correctly said `not_end`

These are the most useful qualitative improvements.

In [ ]:
# comparison = (
#     baseline_test_results[["conversation_id", "turn_id", "gold", "pred"]]
#     .rename(columns={"pred": "baseline_pred"})
#     .merge(
#         ft_test_results[["conversation_id", "turn_id", "pred"]],
#         on=["conversation_id", "turn_id"],
#         how="inner",
#     )
#     .rename(columns={"pred": "ft_pred"})
# )

# improved = comparison[
#     (comparison["baseline_pred"] != comparison["gold"]) &
#     (comparison["ft_pred"] == comparison["gold"])
# ].copy()

# improved

## Recommended production integration

After fine-tuning, keep the Exit Advisor small.

Recommended runtime flow:
1. Main Agent sends full chat history to Exit Advisor
2. Exit Advisor returns only `end` or `not_end`
3. If `end`:
   - main agent ends the session
   - use a small deterministic final-message template or a tiny separate prompt
4. If `not_end`:
   - main agent continues routing to schedule/info logic

That is cleaner than asking the fine-tuned model to do both classification and rich wording.

In [ ]:
def run_exit_classifier(client, chat_history, model_name):
    label = classify_exit(
        model_name=model_name,
        instructions=EXIT_ADVISOR_BASELINE_INSTRUCTIONS,
        history_turns=chat_history,
    )
    return label

def build_end_message(chat_history):
    return "Thank you for the update. Wishing you all the best."

# In main agent:
# exit_label = run_exit_classifier(...)
# if exit_label == "end":
#     assistant_message = build_end_message(chat_history)
#     end_session = True

## What to do after first fine-tune

If the first model is not good enough:
- inspect test mistakes
- add more examples for:
  - "please stop texting me"
  - "not interested anymore"
  - "booked and mutually closing"
  - "candidate asked a question after scheduling" → often **not** end yet
- keep labels consistent
- keep output format exactly the same
- run the same held-out test again

That loop follows the recommended order:
start with evals, improve data quality first, then iterate on data quantity and only later on hyperparameters.

## Practical note

In this chat, I have `sms_conversations.json` and the project instructions, but I do **not** have your current `exit_agent.py` file itself.

So this notebook is prepared to:
- import it automatically if it exists in your local project structure
- otherwise let you paste the current prompt into one cell

That is the safest way to keep the notebook aligned with your current codebase.